# doctor_prevention_set_properties_orchestrator
## Purpose
Iterates all tables in a Lakehouse and calls `doctor_prevention_set_table_properties` for
each, applying the correct Delta table properties for the given medallion layer. Run
this once when onboarding a Lakehouse to delta-doctor, or after adding a batch of
new tables.
## What it does
- Enumerates all tables via the OneLake ABFSS path — handles both schema-enabled and
  non-schema Lakehouses automatically
- Calls `doctor_prevention_set_table_properties` for each table, passing the Lakehouse GUID,
  table name, schema name, and layer
- Catches and logs errors per table — one failing table does not stop the run
- Prints a summary of tables updated and errored
- Does not apply liquid clustering — cluster key selection is per-table and must be
  set via `doctor_prevention_set_table_properties` directly
## When to use this vs doctor_prevention_set_table_properties
Use this orchestrator to initialise an entire Lakehouse in one pipeline step. Once
tables are configured, prefer calling `doctor_prevention_set_table_properties` individually
when adding new tables — there is no need to re-run the full Lakehouse on every change.
## One Lakehouse per layer
This notebook assumes one Lakehouse per medallion layer, which is the standard Fabric
pattern. Run it once per Lakehouse, passing the matching `layer` parameter each time:
- Bronze Lakehouse → `layer = "bronze"`
- Silver Lakehouse → `layer = "silver"`
- Gold Lakehouse   → `layer = "gold"`
## Note on the workspace
`mssparkutils.notebook.run()` identifies the target notebook by name within the same
Fabric workspace. Ensure `doctor_prevention_set_table_properties` has been imported into
the same workspace as this orchestrator before running. Both notebooks must be in the
same workspace as the target Lakehouse.
## Timeout
Each child notebook call uses `timeout_seconds=120`. Property-setting is metadata-only —
120 seconds is sufficient for any OneLake-reachable table. A timeout indicates an
OneLake connectivity issue, not a problem with the notebook itself.
## Warning — deletion vectors protocol upgrade
This notebook enables `delta.enableDeletionVectors = true` on **every table** in the
Lakehouse in a single run. This upgrades the Delta reader/writer protocol on each table —
the upgrade is **not reversible** on most table types without rewriting the table.
Clients known not to support deletion vectors include:
- Azure Synapse Analytics external tables
- Databricks runtimes prior to 11.3 LTS
- Some third-party BI connectors and JDBC drivers
If any external clients read tables in this Lakehouse, verify compatibility first. For
selective application, use `doctor_prevention_set_table_properties` directly on individual
tables — it gives you per-table control over whether deletion vectors are enabled.


In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
# These values are overridden at runtime by the Fabric pipeline.
# Default values below are used when running the notebook interactively.

lakehouse_guid = ""        # The GUID of the Lakehouse to configure
layer          = "silver"  # Medallion layer: "bronze", "silver", or "gold". "custom" is not supported — use doctor_prevention_set_table_properties directly for tables requiring non-standard configuration

## Parameters
| Parameter | Type | Description |
|---|---|---|
| `lakehouse_guid` | string | The GUID of the Lakehouse to configure. Found in the Lakehouse URL in the Fabric portal |
| `layer` | string | The medallion layer for all tables in this Lakehouse. Accepts `"bronze"`, `"silver"`, or `"gold"`. `"custom"` is not supported — use `doctor_prevention_set_table_properties` directly for tables requiring non-standard configuration. Default: `"silver"` |


## Validation


In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────

valid_layers = {"bronze", "silver", "gold"}

if not lakehouse_guid:
    raise ValueError("Parameter 'lakehouse_guid' is required but was not provided.")

if not layer or layer.lower() not in valid_layers:
    raise ValueError(f"Parameter 'layer' must be one of: {', '.join(sorted(valid_layers))}. Got: '{layer}'")

layer = layer.lower()

workspace_guid = spark.conf.get("trident.workspace.id")

print(f"Lakehouse: {lakehouse_guid}")
print(f"Layer    : {layer}")

## Functions
`list_delta_tables()` enumerates all Delta tables in the Lakehouse via ABFSS path listing,
handling both schema-enabled and non-schema Lakehouses automatically.


In [ ]:
# ── Table enumeration ─────────────────────────────────────────────────────────

def list_delta_tables(workspace_guid, lakehouse_guid):
    """
    Enumerates all Delta tables in a Lakehouse via ABFSS path listing.
    Handles both schema-enabled Lakehouses (Tables/{schema}/{table}) and
    non-schema Lakehouses (Tables/{table}) by checking for _delta_log presence.
    Returns a list of dicts: {"schema": str, "table": str, "path": str}.
    """
    tables_root = f"abfss://{workspace_guid}@onelake.dfs.fabric.microsoft.com/{lakehouse_guid}/Tables"
    result = []
    try:
        top_items = mssparkutils.fs.ls(tables_root)
    except Exception as e:
        raise RuntimeError(f"Could not list Tables directory for Lakehouse {lakehouse_guid}: {e}")

    for item in top_items:
        item_name = item.name.rstrip('/')
        try:
            sub_items = mssparkutils.fs.ls(item.path)
            sub_names = [s.name.rstrip('/') for s in sub_items]
            if "_delta_log" in sub_names:
                result.append({"schema": "", "table": item_name, "path": item.path.rstrip('/')})
            else:
                # Potential schema folder — recurse one level
                for sub_item in sub_items:
                    sub_name = sub_item.name.rstrip('/')
                    try:
                        deep_items = mssparkutils.fs.ls(sub_item.path)
                        deep_names = [d.name.rstrip('/') for d in deep_items]
                        if "_delta_log" in deep_names:
                            result.append({"schema": item_name, "table": sub_name, "path": sub_item.path.rstrip('/')})
                    except Exception as e:
                        print(f"  Warning: could not enumerate {sub_item.path} — skipped ({e})")
        except Exception as e:
            print(f"  Warning: could not enumerate {item.path} — skipped ({e})")
    return result

## Orchestration
Iterates all tables in the Lakehouse and calls `doctor_prevention_set_table_properties` for
each. Errors on individual tables are caught and logged — the run continues regardless.


In [ ]:
# ── Orchestration ─────────────────────────────────────────────────────────────

tables = list_delta_tables(workspace_guid, lakehouse_guid)

updated_count = 0
error_count   = 0

print(f"Tables found: {len(tables)}")
print("Warning: deletion vectors will be enabled on all tables, upgrading the Delta protocol.")
print("         Verify external reader compatibility before proceeding.")
print("-" * 60)

for entry in tables:
    schema_val   = entry["schema"]
    table_name   = entry["table"]
    display_name = f"{schema_val}.{table_name}" if schema_val else table_name
    try:
        mssparkutils.notebook.run(
            "doctor_prevention_set_table_properties",
            timeout_seconds=120,
            arguments={
                "lakehouse_guid": lakehouse_guid,
                "table_name":     table_name,
                "schema_name":    schema_val,
                "layer":          layer,
            }
        )
        print(f"  {display_name}: properties set")
        updated_count += 1
    except Exception as e:
        print(f"  {display_name}: ERROR — {str(e)}")
        error_count += 1

print("-" * 60)
print(f"Summary — updated: {updated_count} | errors: {error_count}")